# Data cleaning before training 

Data quality summary:

1000-level total reviews: 2189

2000-level total reviews: 871

3000-level total reviews: 424

4000-level total reviews: 316

5000-level total reviews: 104


1000-level course with comment count: 14

2000-level course with comment count: 8

3000-level course with comment count: 13

4000-level course with comment count: 37

5000-level course with comment count: 6

Path：\IEDA3560_Project\ust_reviews\{course_name}\{course_name}_reviews.csv

feature number= column number: 30

feature names: ['hash', 'semester', 'instructors', 'is_author', 'author', 'date', 'title', 'comment_content', 'comment_teaching', 'comment_grading', 'comment_workload', 'rating_content', 'rating_teaching', 'rating_grading', 'rating_workload', 'has_midterm', 'has_final', 'has_quiz', 'has_assignment', 'has_essay', 'has_project', 'has_attendance', 'has_reading', 'has_presentation', 'upvote_count', 'vote_count', 'voted', 'is_upvote', 'comment_count', 'attachments']


quality:
A lot of binary variable, with integer, the comments are the mixture of Eng and Chin, with oral expression 
Some unique key can be dropped 


This cleaning is for adding additional data sample


# Fouthrth data cleaning

For each course csv file: in path \IEDA3560_Project\ust_reviews
1. Add the course_name as the tag for each csv fill the column with the column name on its path
2. Add the column: level,  for the fifth digit of the column course name if digit 5 =1 ==> fill:1000  ==2 ==> fill 2000 eg COMP1021 ==> fill 1000
3. Add a column: upvote_ratio, value for each cell  = upvote_count/ vote_count
4. Add a column: insturctor_name, value = get the first json in column instructors, key = name  which is the course insturtor name, we will use it to map professor rating from the raw data from
https://ust-rankings.com/ ,

the cleaned data path \IEDA3560_Project\ust_reviews\instructor_ranking.csv

#  the value in column insturctors = [], still keep and set insturctor_name / insturctor_rating as missing (NaN).

and we create a new column insturctor_rating to store the value after mapping the instructor name and its rating ## to prevent the data leakage, the calculation for the score is not the same as those in ustranking website, as we eliminated the workload rating. 

=  A+ =11 A=10 A-=9 B+=8, B=7, B-=6, C+=5, C=4, C-=3, D=2, E=1, F=0 

5. sort the data with date, only keeping the data within recent 5 years e.g. date = xxx xx, year, year= range[2020,2026]
6. for each course keep at most 50 or(80)recent reviews
7. save a df_course_name with column  [ 'rating_workload','rating_content', 'rating_teaching', 'rating_grading','has_midterm', 'has_final', 'has_quiz', 'has_assignment', 'has_essay', 'has_project', 'has_attendance', 'has_reading', 'has_presentation','course_name,level,'upvote_ratio','insturctor_rating']

 equipvalent to dropping ['hash', 'semester', 'instructors', 'is_author', 'author', 'date', 'title', 'comment_content', 'comment_teaching', 'comment_grading', 'comment_workload', 'upvote_count', 'vote_count', 'voted', 'is_upvote', 'comment_count', 'attachments', insturctor_name]

8. check the data remaining and iterate e.g. check how many value counted in total, if it filter out too much, then adjust the year range 


In [1]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd


In [2]:
# mapping function to define instructor score based on ranking
def build_instructor_score_map(ranking_path):
    rating_map = {
        "A+": 11, "A": 10, "A-": 9,
        "B+": 8, "B": 7, "B-": 6,
        "C+": 5, "C": 4, "C-": 3,
        "D": 2, "E": 1, "F": 0
    }

    rank_df = pd.read_csv(ranking_path, encoding="utf-8-sig")
    rank_df.columns = [c.strip() for c in rank_df.columns]

    if "instructor name" not in rank_df.columns or "ranking" not in rank_df.columns:
        raise ValueError("instructor_ranking.csv must contain: instructor name, ranking")

    rank_df["instructor name"] = rank_df["instructor name"].astype(str).str.strip()
    rank_df["ranking"] = rank_df["ranking"].astype(str).str.strip().str.upper()
    rank_df["insturctor_rating"] = rank_df["ranking"].map(rating_map)

    name_to_score = (
        rank_df.dropna(subset=["instructor name"])
               .drop_duplicates(subset=["instructor name"], keep="first")
               .set_index("instructor name")["insturctor_rating"]
               .to_dict()
    )
    return name_to_score, rank_df

ROOT = Path("./ust_reviews")  # input: raw data reviews
RANKING_PATH = ROOT / "instructor_ranking.csv"
name_to_score, rank_df = build_instructor_score_map(RANKING_PATH)
display(rank_df.head())

,instructor name,ranking,score,insturctor_rating
0,"TSOI, Yau Chat",A+,0.711526,11
1,"FONG, Tsz Ho",A+,0.567629,11
2,"NG, Ka Man",A+,0.528440,11
3,"BAO, Zhigang",A+,0.525558,11
4,"ARYA, Sunil",A+,0.514835,11


In [3]:
name_to_score

{'TSOI, Yau Chat': 11,
 'FONG, Tsz Ho': 11,
 'NG, Ka Man': 11,
 'BAO, Zhigang': 11,
 'ARYA, Sunil': 11,
 'WONG, Raymond C W': 11,
 'NASON, Stephen William': 11,
 'WONG, James K.': 11,
 'IP, Ivan Chi Ho': 11,
 'CHAN, Gary Shueng Han': 11,
 'SHEN, Yiwen': 11,
 'LEUNG, Chi Man': 11,
 'WANG, Xuan': 11,
 'JEONG, Martha': 11,
 'LIEM Rhea P': 11,
 'WONG, Tsz Wai': 11,
 'AU, Pak Hung': 11,
 'LAW, Kam Tuen': 11,
 'TANG, Rui': 11,
 'KAFSHDAR GOHARSHADY, Amir': 11,
 'WONG, Simon Man Ho': 11,
 'SHEONG, Frederick Fu Kit': 11,
 'ROSSITER, David': 11,
 'CHENG, Kam Hang': 11,
 'HUANG, Yong': 11,
 'ENGLAND, Graham James Fisher': 11,
 'MENG, Zili': 11,
 'SIU, Wai Sze Grace': 11,
 'SHIOMI, Koji': 11,
 'KAILA, Ilari Julius': 11,
 'HUA, Xinyu': 11,
 'SIU, Kam Wing': 11,
 'SHIEH, Tony': 11,
 'CHEUNG, Kwok Yip': 11,
 'FU, Li-tsui': 11,
 'SHI, Ling': 11,
 'DALTON, Amy N': 11,
 'LEUNG, Shing Yu': 11,
 'FRANKLIN, Laurence Craig': 11,
 'ZHANG, Hongtao': 11,
 'LI, Larry': 11,
 'PARK, Eunyoung': 11,
 'YU, Yan': 11

In [4]:
# helper function to extract course instructor name from the raw instructor column, which can be a string or a list of dicts
def parse_first_instructor_name(x):
    if pd.isna(x):
        return np.nan

    try:
        value = x

        if isinstance(value, str):
            value = value.strip()
            if not value or value == "[]":
                return np.nan
            value = ast.literal_eval(value)

        if isinstance(value, list) and len(value) > 0:
            first_item = value[0]
            if isinstance(first_item, dict):
                name = first_item.get("name")
                if pd.notna(name) and str(name).strip():
                    return str(name).strip()
            elif isinstance(first_item, str):
                return first_item.strip() or np.nan

        return np.nan
    except Exception:
        return np.nan

In [5]:
# helper function to extract course level from course name 
def get_level(course_name):
    # COMP1021 -> 1000
    try:
        d = str(course_name)[4]
        if d.isdigit():
            return int(d) * 1000
    except Exception:
        pass
    return np.nan

In [6]:
# cleaning for a single course file, return cleaned df and stats
TARGET_COLS = [
    "rating_workload",'rating_content', 'rating_teaching', 'rating_grading',
    "has_midterm", "has_final", "has_quiz", "has_assignment", "has_essay",
    "has_project", "has_attendance", "has_reading", "has_presentation",
    "course_name", "level", "upvote_ratio", "instructor_rating","comment_content","comment_teaching","comment_workload","comment_grading"
]
def clean_one_course_file(
    fp,
    name_to_score,
    min_year=2016,
    max_year=2026,
    output_root=None,
    max_reviews_per_course=80
):
    course_name = fp.stem.replace("_reviews", "")
    df = pd.read_csv(fp, encoding="utf-8-sig")
    input_rows = len(df)

    df["course_name"] = course_name
    df["level"] = get_level(course_name)

    df["upvote_count"] = pd.to_numeric(df.get("upvote_count"), errors="coerce")
    df["vote_count"] = pd.to_numeric(df.get("vote_count"), errors="coerce")
    df["upvote_ratio"] = np.where(
        df["vote_count"].fillna(0) > 0,
        df["upvote_count"].fillna(0) / df["vote_count"],
        np.nan
    )

    df["instructor_name"] = df.get(
        "instructors",
        pd.Series(index=df.index, dtype=object)
    ).apply(parse_first_instructor_name)
# droping logic removed to keep all reviews regardless of whether instructor name is available

    df["instructor_rating"] = df["instructor_name"].map(name_to_score)

    df["date_parsed"] = pd.to_datetime(df.get("date"), errors="coerce") 
    before_drop_year = len(df)
    df = df[df["date_parsed"].dt.year.between(min_year, max_year, inclusive="both")].copy()
    dropped_out_of_year_range = before_drop_year - len(df)

    # Keep at most N most recent reviews per course
    before_cap = len(df)
    df = df.sort_values("date_parsed", ascending=False).head(max_reviews_per_course).copy()
    dropped_by_cap = before_cap - len(df)

    for c in TARGET_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df_out = df[TARGET_COLS].copy()

    if output_root is not None:
        course_out_dir = output_root / course_name
        course_out_dir.mkdir(parents=True, exist_ok=True)
        out_path = course_out_dir / f"{course_name}_cleaned.csv"
        df_out.to_csv(out_path, index=False, encoding="utf-8-sig")

    stats = {
        "course_name": course_name,
        "input_rows": input_rows,
        "dropped_out_of_year_range": dropped_out_of_year_range,
        "dropped_by_cap": dropped_by_cap,
        "output_rows": len(df_out)
    }
    return df_out, stats

In [7]:
# test with comp1021 
MAX_REVIEWS_PER_COURSE = 80
max_reviews_per_course=MAX_REVIEWS_PER_COURSE

fp = ROOT / "COMP1021" / "COMP1021_reviews.csv"
out_dir = Path(r"C:\Users\aazh0\Desktop\IEDA3560_Project\ust_review_cleanned_with_reviews_10yrs_no_instructor_dropping")
out_dir.mkdir(parents=True, exist_ok=True)

df_comp1021_cleaned, stats_comp1021 = clean_one_course_file(
    fp=fp,
    name_to_score=name_to_score,
    min_year=2016,
    max_year=2026,
    output_root=out_dir
)

df_comp1021_cleaned.head()

,rating_workload,rating_content,rating_teaching,rating_grading,has_midterm,has_final,has_quiz,has_assignment,has_essay,has_project,...,has_reading,has_presentation,course_name,level,upvote_ratio,instructor_rating,comment_content,comment_teaching,comment_workload,comment_grading
115,5,5,5,5,True,True,False,False,False,False,...,False,False,COMP1021,1000,NaN,11.0,The content is mainly the basic python program...,"The teaching is fair, if you like to attend th...","The workload is fair, about 1 PA a week, the P...","The grading is fair, since most of the student..."
116,4,3,4,3,True,True,False,True,False,False,...,False,False,COMP1021,1000,NaN,11.0,"Exploring the content, the following topics ar...",David&#039;s teaching was great!,"Workload is average as a required course, typi...","- Midterm 24%, Lab projects 36%, Final exam 40..."
42,4,5,4,4,True,True,False,True,False,False,...,False,False,COMP1021,1000,1.0,NaN,内容充分有趣，說真的，這老師上課是真的有點東西。<br /><br />我見過的老師多了，能...,NaN,NaN,NaN
27,5,4,3,5,True,True,False,True,False,False,...,False,False,COMP1021,1000,1.0,NaN,對於有學過python嘅人嚟講唔難<br />有考過ICT DSE嘅應該都ok，只係比DSE...,冇認真上堂，評價唔到<br />唔計attendance,open book exam，不過背齊格式其實冇乜好睇<br />lab平均用時一個鐘，唔難,midterm同final exam 都好大SD，around 20<br />Midter...
117,5,5,5,5,True,True,False,True,False,False,...,False,False,COMP1021,1000,NaN,11.0,pretty easy to follow but the difficulty of as...,very clear but he speaks very fastly,really light workload.<br />This lecture has 3...,If you dedicate even a modest amount of consis...


In [8]:
stats_comp1021

{'course_name': 'COMP1021',
 'input_rows': 1234,
 'dropped_out_of_year_range': 26,
 'dropped_by_cap': 1128,
 'output_rows': 80}

In [9]:
MIN_YEAR = 2016
MAX_YEAR = 2026
OUTPUT_ROOT = Path(r"C:\Users\aazh0\Desktop\IEDA3560_Project\ust_review_cleanned_with_reviews_10yrs_no_instructor_dropping")  # output: 清洗後資料

In [10]:
def clean_all_courses(root=ROOT, name_to_score=None, min_year=MIN_YEAR, max_year=MAX_YEAR, output_root=OUTPUT_ROOT):
    files = sorted(root.glob("COMP*/COMP*_reviews.csv"))
    if not files:
        raise FileNotFoundError("No course review csv found under ./ust_reviews")
    if name_to_score is None:
        raise ValueError("name_to_score is required")

    all_cleaned = []
    report_rows = []

    for fp in files:
        df_out, stats = clean_one_course_file(
            fp=fp,
            name_to_score=name_to_score,
            min_year=min_year,
            max_year=max_year,
            output_root=output_root
        )
        all_cleaned.append(df_out)
        report_rows.append(stats)

    df_all_cleaned = pd.concat(all_cleaned, ignore_index=True)
    report_df = pd.DataFrame(report_rows).sort_values("output_rows", ascending=False)

    df_all_cleaned.to_csv(output_root / "all_courses_cleaned.csv", index=False, encoding="utf-8-sig")
    report_df.to_csv(output_root / "cleaning_report.csv", index=False, encoding="utf-8-sig")

    return df_all_cleaned, report_df

In [11]:
MAX_REVIEWS_PER_COURSE = 80
max_reviews_per_course=MAX_REVIEWS_PER_COURSE
ROOT = Path("./ust_reviews")  # input: raw data reviews
OUTPUT_ROOT = Path(r"C:\Users\aazh0\Desktop\IEDA3560_Project\ust_review_cleanned_with_reviews_10yrs_no_instructor_dropping")  # output: 清洗後資料
RANKING_PATH = ROOT / "instructor_ranking.csv"

MIN_YEAR = 2016
MAX_YEAR = 2026

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [12]:
df_all_cleaned, report_df = clean_all_courses(
    root=ROOT,
    name_to_score=name_to_score,
    min_year=MIN_YEAR,
    max_year=MAX_YEAR,
    output_root=OUTPUT_ROOT
)

total_input = report_df["input_rows"].sum()
total_drop_year = report_df["dropped_out_of_year_range"].sum()
total_output = report_df["output_rows"].sum()
total_drop_cap = report_df["dropped_by_cap"].sum()


print("==== Cleaning Summary ====")
print("Total input rows:", total_input)
print(f"Dropped due to year filter [{MIN_YEAR}, {MAX_YEAR}]:", total_drop_year)
print(f"Dropped due to per-course cap ({MAX_REVIEWS_PER_COURSE}):", total_drop_cap)
print("Total output rows:", total_output)
print("Retention ratio:", f"{(total_output / total_input):.2%}" if total_input > 0 else "N/A")

display(report_df.head(20))
display(df_all_cleaned.head(20))

==== Cleaning Summary ====
Total input rows: 3904
Dropped due to year filter [2016, 2026]: 102
Dropped due to per-course cap (80): 1866
Total output rows: 1936
Retention ratio: 49.59%


,course_name,input_rows,dropped_out_of_year_range,dropped_by_cap,output_rows
2,COMP1022P,356,12,264,80
3,COMP1022Q,231,18,133,80
15,COMP2012,109,4,25,80
1,COMP1021,1234,26,1128,80
14,COMP2011,311,7,224,80
20,COMP2711,177,5,92,80
0,COMP1001,74,2,0,72
10,COMP1942,73,1,0,72
16,COMP2012H,75,3,0,72
32,COMP3711,73,2,0,71


,rating_workload,rating_content,rating_teaching,rating_grading,has_midterm,has_final,has_quiz,has_assignment,has_essay,has_project,...,has_reading,has_presentation,course_name,level,upvote_ratio,instructor_rating,comment_content,comment_teaching,comment_workload,comment_grading
0,4,5,4,4,False,True,True,False,False,True,...,False,False,COMP1001,1000,NaN,0.0,The course is relatively easy and feels very s...,"Not that important, as mentioned lecture conte...","Weekly Labs Assignment: straightforward, can b...",Grade Weighting:<br />Lab Exam: 40%<br />Proje...
1,3,3,3,3,True,True,False,False,False,True,...,False,False,COMP1001,1000,NaN,0.0,"The course is very beginner friendly, and disc...","Unlike other classes, the lectures aren&#039;t...","Workload is manageable, just scope the final p...",Grading is fair: multiple choice quizzes about...
2,5,5,4,5,False,True,True,False,False,True,...,True,True,COMP1001,1000,NaN,0.0,Content covers a few basic useful software lik...,ok ge,"not much workload, a bit busy near the project...","Not sure about the grading, but since most stu..."
3,5,5,5,5,False,True,True,False,False,True,...,False,True,COMP1001,1000,NaN,0.0,中學ICT野，讀過嘅基本上唔使睇lecture notes<br />冇讀都唔緊要，只係肯花...,"冇睇過佢啲lec video, 聽人講佢教書係麻麻嘅，但係呢科嘅內容應該唔需要睇video,...","每星期有一份5題嘅lecture quiz mc, 可以睇住notes做，只要夠小心就一定滿...","好好grade, 就算唔認真讀，只係肯花時間整proj都一定有b range<br />應該..."
4,4,5,5,5,False,True,True,True,False,True,...,False,True,COMP1001,1000,NaN,0.0,Content covers a few useful software like all ...,I think the teaching style is unique and quite...,Workload may be slightly heavy to create a goo...,"Overall, I think it&#039;s great. As long as y..."
5,4,4,5,5,False,True,False,True,False,True,...,False,False,COMP1001,1000,NaN,0.0,Manageable for those who have no IT background...,"Great professor, the TA is so nice as well",Quizzes on Canvas (easy)<br />Project (better ...,Exam: 30% (online exam)<br />Project: 40%<br /...
6,5,4,4,4,True,True,False,True,False,False,...,False,False,COMP1001,1000,0.000000,NaN,"As an introduction to programming, this course...","Attendance is not counted in the course, so I ...",4 lab(40%)(not very difficult)<br />mid term (...,"B(4 lab full marks, final below median)"
7,4,4,4,4,False,True,True,True,False,True,...,False,False,COMP1001,1000,NaN,NaN,Lec: <br />Attendance will be counted<br /><br...,The professor has a good accent and you can he...,Project:<br />The project is the biggest workl...,Project: 40% <br />Exam: 30%<br />Labs: 10%<br...
8,4,4,5,4,False,True,True,False,False,True,...,False,False,COMP1001,1000,NaN,0.0,Some mc questions about DSE ICT content.<br />...,Good teaching.<br />Lecture activities are funny.,Some wordload. You need to work harder on the ...,It is an easy course so not that hard to get a...
9,5,4,5,5,False,True,True,True,False,True,...,False,True,COMP1001,1000,0.500000,0.0,very simple topics<br />helps you to gain some...,"one of the most-friendly, fun professor I&#039...","personally, I feel like this is the best cours...",[in-class activity bonus]<br />but it can be p...


In [13]:
report_df.head(20)

,course_name,input_rows,dropped_out_of_year_range,dropped_by_cap,output_rows
2,COMP1022P,356,12,264,80
3,COMP1022Q,231,18,133,80
15,COMP2012,109,4,25,80
1,COMP1021,1234,26,1128,80
14,COMP2011,311,7,224,80
20,COMP2711,177,5,92,80
0,COMP1001,74,2,0,72
10,COMP1942,73,1,0,72
16,COMP2012H,75,3,0,72
32,COMP3711,73,2,0,71


In [14]:
df_all_cleaned.head()

,rating_workload,rating_content,rating_teaching,rating_grading,has_midterm,has_final,has_quiz,has_assignment,has_essay,has_project,...,has_reading,has_presentation,course_name,level,upvote_ratio,instructor_rating,comment_content,comment_teaching,comment_workload,comment_grading
0,4,5,4,4,False,True,True,False,False,True,...,False,False,COMP1001,1000,NaN,0.0,The course is relatively easy and feels very s...,"Not that important, as mentioned lecture conte...","Weekly Labs Assignment: straightforward, can b...",Grade Weighting:<br />Lab Exam: 40%<br />Proje...
1,3,3,3,3,True,True,False,False,False,True,...,False,False,COMP1001,1000,NaN,0.0,"The course is very beginner friendly, and disc...","Unlike other classes, the lectures aren&#039;t...","Workload is manageable, just scope the final p...",Grading is fair: multiple choice quizzes about...
2,5,5,4,5,False,True,True,False,False,True,...,True,True,COMP1001,1000,NaN,0.0,Content covers a few basic useful software lik...,ok ge,"not much workload, a bit busy near the project...","Not sure about the grading, but since most stu..."
3,5,5,5,5,False,True,True,False,False,True,...,False,True,COMP1001,1000,NaN,0.0,中學ICT野，讀過嘅基本上唔使睇lecture notes<br />冇讀都唔緊要，只係肯花...,"冇睇過佢啲lec video, 聽人講佢教書係麻麻嘅，但係呢科嘅內容應該唔需要睇video,...","每星期有一份5題嘅lecture quiz mc, 可以睇住notes做，只要夠小心就一定滿...","好好grade, 就算唔認真讀，只係肯花時間整proj都一定有b range<br />應該..."
4,4,5,5,5,False,True,True,True,False,True,...,False,True,COMP1001,1000,NaN,0.0,Content covers a few useful software like all ...,I think the teaching style is unique and quite...,Workload may be slightly heavy to create a goo...,"Overall, I think it&#039;s great. As long as y..."


In [15]:
total_rows = len(df_all_cleaned)
nan_upvote_ratio_count = df_all_cleaned["upvote_ratio"].isna().sum()

print("Total rows:", total_rows)
print("upvote_ratio is NaN:", nan_upvote_ratio_count)

Total rows: 1936
upvote_ratio is NaN: 739


In [16]:
level1_df = df_all_cleaned[df_all_cleaned["level"] == 1000].copy()
print("Level 1000 rows:", len(level1_df))
level2_df = df_all_cleaned[df_all_cleaned["level"] == 2000].copy()
print("Level 2000 rows:", len(level2_df))
level3_df = df_all_cleaned[df_all_cleaned["level"] == 3000].copy()
print("Level 3000 rows:", len(level3_df))
level4_df = df_all_cleaned[df_all_cleaned["level"] == 4000].copy()
print("Level 4000 rows:", len(level4_df))
level5_df = df_all_cleaned[df_all_cleaned["level"] == 5000].copy()
level5_df.head()
print("Level 5000 rows:", len(level5_df))

Level 1000 rows: 595
Level 2000 rows: 507
Level 3000 rows: 418
Level 4000 rows: 312
Level 5000 rows: 104


In [17]:
# check review distribution over the course level 
level_course_counts = (
    df_all_cleaned
    .groupby(["level", "course_name"], as_index=False)
    .size()
    .rename(columns={"size": "review_count"})
    .sort_values(["level", "review_count"], ascending=[True, False])
)

level_course_counts[level_course_counts["level"] == 1000] 

,level,course_name,review_count
1,1000,COMP1021,80
2,1000,COMP1022P,80
3,1000,COMP1022Q,80
0,1000,COMP1001,72
10,1000,COMP1942,72
4,1000,COMP1023,63
6,1000,COMP1029C,31
12,1000,COMP1944,31
7,1000,COMP1029J,24
11,1000,COMP1943,22


In [18]:
level_course_counts[level_course_counts["level"] == 2000]

,level,course_name,review_count
14,2000,COMP2011,80
15,2000,COMP2012,80
20,2000,COMP2711,80
16,2000,COMP2012H,72
21,2000,COMP2711H,65
17,2000,COMP2211,60
18,2000,COMP2611,59
19,2000,COMP2633,11


In [19]:
level_course_counts[level_course_counts["level"] == 3000]

,level,course_name,review_count
32,3000,COMP3711,71
28,3000,COMP3511,67
24,3000,COMP3111,61
22,3000,COMP3021,50
27,3000,COMP3311,48
23,3000,COMP3031,28
26,3000,COMP3211,24
34,3000,COMP3721,22
33,3000,COMP3711H,17
30,3000,COMP3632,14


In [20]:
level_course_counts[level_course_counts["level"] == 4000]

,level,course_name,review_count
37,4000,COMP4211,34
35,4000,COMP4021,23
46,4000,COMP4441,20
45,4000,COMP4431,17
41,4000,COMP4331,16
44,4000,COMP4421,15
62,4000,COMP4651,14
38,4000,COMP4221,13
42,4000,COMP4332,11
43,4000,COMP4411,11


In [21]:
level_course_counts[level_course_counts["level"] == 5000]

,level,course_name,review_count
73,5000,COMP5212,43
76,5000,COMP5621,34
72,5000,COMP5211,16
77,5000,COMP5631,6
75,5000,COMP5311,4
74,5000,COMP5221,1
